# Visual ML Foundations: 2.5D Slice Stacking for Volumetric Imaging
### Bridging 2D Pretrained Backbones (ImageNet / DINOv2) to 3D Medical Scans

---

## 1. Concept Overview
Full 3D convolutional networks require massive GPU VRAM ($>16\text{ GB}$ per small batch) and lack massive pre-trained weights.
The **2.5D Slice Stacking** technique solves this:
* Standard vision backbones (ResNet, DINOv2, ConvNeXt) expect 3 RGB color channels: $(3, H, W)$.
* Instead of RGB, we stack 3 adjacent anatomical slices:
  $$\text{Channel } 0 = \text{Slice } i-1, \quad \text{Channel } 1 = \text{Slice } i, \quad \text{Channel } 2 = \text{Slice } i+1$$
* This gives the 2D network local 3D through-plane context while running at 10x faster inference speed!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate 3 consecutive Sagittal MRI slices showing meniscus tear evolution
h, w = 120, 120
y, x = np.ogrid[:h, :w]

slice_prev = np.zeros((h, w))
slice_curr = np.zeros((h, w))
slice_next = np.zeros((h, w))

# Bone background
bone = ((y - 45)**2 + (x - 60)**2) < 26**2
for s in [slice_prev, slice_curr, slice_next]:
    s[bone] = 0.75

# Joint space with evolving meniscus tear
slice_prev[68:74, 50:70] = 0.35 # Mild signal
slice_curr[67:75, 48:72] = 0.90 # Focal high-signal tear!
slice_next[68:74, 52:68] = 0.40 # Tear fading out

# Stack into 2.5D 3-Channel Tensor
stack_2p5d = np.stack([slice_prev, slice_curr, slice_next], axis=-1)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(slice_prev, cmap='bone')
axes[0].set_title("Channel 0: Slice (i-1)", fontsize=11, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(slice_curr, cmap='bone')
axes[1].set_title("Channel 1: Slice (i) [Tear]", fontsize=11, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(slice_next, cmap='bone')
axes[2].set_title("Channel 2: Slice (i+1)", fontsize=11, fontweight='bold')
axes[2].axis('off')

axes[3].imshow(stack_2p5d)
axes[3].set_title("Combined 2.5D Composite", fontsize=11, fontweight='bold')
axes[3].axis('off')

plt.tight_layout()
plt.show()
print(f"2.5D Tensor shape: {stack_2p5d.shape} (Ready for pre-trained vision backbones)")
